In [1]:
pip install streamlit scikit-learn seaborn joblib matplotlib

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   - -------------------------------------- 0.4/15.5 MB 8.5 MB/s eta 0:00:02
   - -------------------------------------- 0.7/15.5 MB 8.9 MB/s eta 0:00:02
   --- ------------------------------------ 1.4/15.5 MB 9.9 MB/s eta 0:00:02
   ---- ----------------------------------- 1.9/15.5 MB 10.7 MB/s eta 0:00:02
   ------ --------------------------------- 2.5/15.5 MB 10.7 MB/s eta 0:00:02
   ------- -------------------------------- 3.1/15.5 MB 10.8 MB/s eta 0:00:02
   --------- ------------------------------ 3.6/15.5 MB 10.8 MB/s eta 0:00:02
   ---------- ----------------------------- 4.1/15.5 MB 10.8 MB/s eta 0:00:02
   ----------- ---------------------------- 4.6/15.5 MB 10.9 MB/s eta 0:00:01
   ------------- -------------------------- 5.2/15.5 MB 11.0 MB/s eta 0:00:01
 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [2]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import numpy as np
import matplotlib.pyplot as plt
import os 
import json 

df = sns.load_dataset('titanic')

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
features = ["pclass","sex","age","sibsp","parch","fare","embarked"]
df = df[features+["survived"]]


,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0


In [ ]:
df.isna().sum()

pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

In [12]:
df["age"]= df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna("S")

In [11]:
df["embarked"].value_counts()

embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [13]:
df.isna().sum()

pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64

In [14]:
X= df[features]
y = df["survived"]


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=23, stratify=y)


In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),["sex","embarked"]),
        ("num","passthrough",["pclass","age","sibsp","parch","fare"])
    ]
)

In [17]:
pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(n_estimators=100,random_state=23))
])

In [18]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['sex', 'embarked']),
                                                 ('num', 'passthrough',
                                                  ['pclass', 'age', 'sibsp',
                                                   'parch', 'fare'])])),
                ('classifier', RandomForestClassifier(random_state=23))])

In [26]:
y_pred = pipeline.predict(X_test)
y_proba =pipeline.predict_proba(X_test)[:,1]
report = classification_report(y_test,y_pred, output_dict=True)
auc = float(roc_auc_score(y_test,y_proba))
cm = confusion_matrix(y_test,y_pred).tolist()

In [28]:
os.makedirs("artifacts",exist_ok=True)
joblib.dump(pipeline,"artifacts/titanic_model.pkl")

['artifacts/titanic_model.pkl']

In [29]:
with open("artifacts/metric.json","w") as f:
    json.dump({"classification_report":report, "roc_auc_score":auc, "confusion_matrix":cm},f, indent=2)

In [32]:
fig, ax = plt.subplots(figsize=(4,4))
ax.matshow(confusion_matrix(y_test,y_pred), cmap="Blues", alpha=0.7)
for (i,j), z in np.ndenumerate(confusion_matrix(y_test,y_pred)):
    ax.text(j, i, str(z), ha="center",va="center")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.title(f"Confusion matrix (AUC={auc:.3f})")
plt.savefig("artifacts/confusion_matrix.png",bbox_inches="tight")
plt.close()